# Credit Call Spread — Momentum Reversal Vol Premium Capture

This strategy sells a call spread on SPY following a 1-day up-move while posting a negative return on a 15-day basis, fading the short-term bounce within a negative momentum regime. The hypothesis is that speculative call buying on bounces in a downtrend creates a temporary bid in implied volatility on the short end of the term structure. This premium dislocation is monetized by targeting floating delta strikes with a positive IV z-score relative to their 90-day history, 
filtered by a positive VRP at the selected expiry.

![QuantConnect Logo](https://cdn.quantconnect.com/web/i/icon.png)
<hr>

## Methodology

**Data & Signal**
Daily SPY price history is used to construct a 15-day momentum signal and 22-day realized volatility. 
A trade signal fires when SPY posts a positive single-day return within a negative 15-day momentum regime, 
identifying short-term bounces within a broader downtrend.

**Expiry Selection**
For each signal date, ATM implied volatility is pulled across all SPY option expiries within an 8-week 
window with a minimum of 14 DTE. The expiry with the highest positive volatility risk premium 
(implied vol minus realized vol) is selected as the target expiry.

**Strike Selection**
The short leg targets the floating delta strike with the highest implied volatility z-score relative 
to its 90-day history across the 0.20-0.30 delta range, identifying where the market is paying the 
most above normal levels. The long leg is fixed at the nearest 0.15 delta call on the same expiry.

**Exit Rules**
Positions are closed when any of the following conditions are met:
- DTE falls below 10
- 70% of premium has been captured (spread value < 30% of original premium collected)
- Short call moves in the money (SPY closes above short strike)

In [ ]:
from AlgorithmImports import *
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import timedelta

qb = QuantBook()

START   = datetime(2024, 1, 1)
END     = datetime(2024, 10, 1)

In [166]:
spy = qb.add_equity("SPY", data_normalization_mode=DataNormalizationMode.RAW).symbol
single_history_df = qb.history(TradeBar, spy, START, END,  Resolution.DAILY)
spy_df = single_history_df.loc['SPY']

option = qb.add_option(spy)

spy_df['pct_chg'] = spy_df.close.pct_change()
spy_df['mtum_15d'] = spy_df.close.pct_change(15)
spy_df['log_ret'] = np.log(spy_df.close / spy_df.close.shift(1))
spy_df["rvol_22d"] = spy_df.log_ret.rolling(window=22).std() * np.sqrt(252) * 100
spy_df['rvol_22d'] = spy_df.rvol_22d.apply(lambda x: round(x,2))

spy_df.drop(columns=["high", "low", "open", "volume"], inplace=True)

spy_df = spy_df.dropna()

spy_df.index = pd.to_datetime(spy_df.index).date

trade_date_index = spy_df.index



In [167]:
def get_term_structure(dt: datetime, option=option) -> pd.DataFrame:
    today = pd.Timestamp(dt)
    chain_history = qb.history(
        option.symbol, 
        dt, 
        dt + timedelta(days=1), 
        flatten=True
    ).droplevel([0])

    chain_history['expiry'] = chain_history.index.map(lambda symbol: symbol.ID.Date)
    chain_history['right'] = chain_history.index.map(lambda symbol: symbol.id.option_right)

    filtered_chain = chain_history[
        (chain_history.delta > 0.49) & 
        (chain_history.delta < 0.51)
    ]

    term_struct = filtered_chain[filtered_chain.expiry < pd.Timestamp(dt + timedelta(weeks=8))]
    term_struct = term_struct[(term_struct.expiry - pd.Timestamp(dt)).dt.days >= 14]

    term_struct = term_struct.reset_index()
    term_struct = term_struct[['expiry', 'impliedvolatility']]

    term_struct = term_struct.rename(columns={'impliedvolatility' : 'impvol'})
    term_struct['impvol'] = term_struct.impvol.apply(lambda x: x * 100).round(2)

    term_struct['expiry'] = term_struct['expiry'].dt.date

    return term_struct

def select_expiry(dt: datetime, realized_vol):
    df = get_term_structure(dt)
    df['vrp'] = df.impvol - realized_vol
    df = df[df['vrp'] > 0]
    if df.empty:
        return None
    max_row_index = df['vrp'].idxmax()
    highest_value_date = df.loc[max_row_index, 'expiry']

    return highest_value_date

def highest_zscore_delta(today, option=option):
    today = pd.Timestamp(today)
    chain_history = qb.history(
        option.symbol, 
        today - timedelta(days=90), 
        today, 
        flatten=True
    )
    
    if chain_history.empty:
        return None

    symbols = chain_history.index.get_level_values('symbol')
    chain_history['right'] = [s.id.option_right for s in symbols]
    
    call_history = chain_history[chain_history.right == 'Call']
    call_history = call_history[(call_history.delta <= 0.35) & (call_history.delta >= 0.20)]
    call_history = call_history[['delta', 'impliedvolatility']]

    
    
    delta_values = [0.30, 0.25, 0.20]

    delta_zscore = {}
    for delta in delta_values:

        def find_delta(frame, d=delta):
            delta_diff = (frame['delta'] - d).abs()
            return frame.loc[delta_diff.idxmin()]

        delta_daily = call_history.groupby(level='time').apply(find_delta, include_groups=False)
        arry = delta_daily.impliedvolatility.to_numpy()
        zscore = (arry[-1] - arry.mean()) / arry.std()
        delta_zscore[delta] = zscore
    
    highest_zscore_delta = max(delta_zscore, key=delta_zscore.get)
    
    return highest_zscore_delta

def get_short_leg(today, expiry, option=option):
    today = pd.Timestamp(today)
    chain_history = qb.history(
        option.symbol, 
        today, 
        today + timedelta(days=1), 
        flatten=True
    ).droplevel([0])

    chain_history['expiry'] = chain_history.index.map(lambda symbol: symbol.ID.Date)
    chain_history['expiry'] = pd.to_datetime(chain_history['expiry']).dt.date
    chain_history['right'] = chain_history.index.map(lambda symbol: symbol.id.option_right)

    chain_history = chain_history[chain_history.expiry == expiry]
    chain_history = chain_history[chain_history.right == 'Call']

    if chain_history.empty:
        return None

    zscore_delta = highest_zscore_delta(today=today)
    if zscore_delta is None:
        return None

    closest_idx = (chain_history.delta - zscore_delta).abs().idxmin()
    chain_history = chain_history.loc[[closest_idx]]

    return chain_history

def get_long_leg(today, expiry, option=option):
    today = pd.Timestamp(today)
    chain_history = qb.history(
        option.symbol, 
        today, 
        today + timedelta(days=1), 
        flatten=True
    ).droplevel([0])

    chain_history['expiry'] = chain_history.index.map(lambda symbol: symbol.ID.Date)
    chain_history['expiry'] = pd.to_datetime(chain_history['expiry']).dt.date
    chain_history['right'] = chain_history.index.map(lambda symbol: symbol.id.option_right)
    chain_history = chain_history[chain_history.right == 'Call']

    chain_history = chain_history[chain_history.expiry == expiry]

    if chain_history.empty:  
        return None

    closest_idx = (chain_history.delta - 0.15).abs().idxmin()
    chain_history = chain_history.loc[[closest_idx]]

    return chain_history


In [168]:
open_trade = None
trades = []

for trade_dt in trade_date_index:
    row = spy_df.loc[trade_dt]
    
    if open_trade is None:
        if row['pct_chg'] > 0 and row['mtum_15d'] < 0:
            expiry = select_expiry(dt=trade_dt, realized_vol=row['rvol_22d'])
            if expiry is None:
                continue
            try:
                short_leg = get_short_leg(today=trade_dt, expiry=expiry)
            except Exception as e:
                continue
            if short_leg is None:
                continue
            long_leg = get_long_leg(today=trade_dt, expiry=expiry)
            if long_leg is None:
                continue
            if short_leg.index[0] == long_leg.index[0]:
                continue
            open_trade = {
                'entry_date': trade_dt,
                'short_call_symbol': short_leg.index[0].value,
                'short_call_symbol_obj': short_leg.index[0],
                'long_call_symbol': long_leg.index[0].value,
                'long_call_symbol_obj': long_leg.index[0],
                'expiry': expiry,
                'premium_collected': (short_leg.close.iloc[0] - long_leg.close.iloc[0]),
                'short_strike': short_leg.index[0].ID.StrikePrice,
            }
    else:
        dte = (open_trade['expiry'] - trade_dt).days

        dt_start = pd.Timestamp(trade_dt)
        dt_end = pd.Timestamp(trade_dt + timedelta(days=1))

        short_history = qb.history(open_trade['short_call_symbol_obj'], dt_start, dt_end, Resolution.DAILY)
        long_history = qb.history(open_trade['long_call_symbol_obj'], dt_start, dt_end, Resolution.DAILY)

        if short_history.empty or long_history.empty:
            continue

        short_price = short_history.droplevel([0,1,2,3])['close'].iloc[0]
        long_price = long_history.droplevel([0,1,2,3])['close'].iloc[0]
        current_spread_value = short_price - long_price

        if open_trade['premium_collected'] == 0:
            continue
        if short_price <= 0 or long_price <= 0:
            continue

        if (dte < 10) or (current_spread_value/open_trade['premium_collected'] < 0.3) or (row['close'] > open_trade['short_strike']):
            print(f"{trade_dt} | dte:{dte} | pct_remaining:{current_spread_value/open_trade['premium_collected']:.2f} | itm:{row['close'] > open_trade['short_strike']} | strike:{open_trade['short_strike']} | spy:{row['close']}")
            closed_trade = {}
            closed_trade['pnl'] = open_trade['premium_collected'] - current_spread_value
            closed_trade['premium_at_open'] = open_trade['premium_collected']
            closed_trade['entry_date'] = open_trade['entry_date']
            closed_trade['closed_date'] = trade_dt
            closed_trade['short_leg'] = open_trade['short_call_symbol']
            closed_trade['long_leg'] = open_trade['long_call_symbol']
            trades.append(closed_trade)
            open_trade = None

trades_df = pd.DataFrame(trades)
trades_df['days_held'] = (trades_df['closed_date'] - trades_df['entry_date']).apply(lambda x: x.days)
trades_df['pct_captured'] = trades_df['pnl'] / trades_df['premium_at_open']

## Results

Backtest period: January 2024 — October 2024 (5 trades after filters)

| Metric | Value |
|--------|-------|
| Total Trades | 5 |
| Mean PnL | -$1.15 |
| Max Loss | -$2.86 |
| Win Rate | 40% |
| Mean Days Held | 9 |

2024 presented a challenging environment for this strategy — SPY trended strongly upward for most 
of the year, causing short call strikes to be breached on sustained rallies rather than mean-reverting 
bounces. This is consistent with the strategy's design: it is intended for choppy, mean-reverting 
regimes rather than persistent directional trends.

## Limitations & Next Steps

- Single position model — a production implementation would include continuous re-entry logic
- No transaction cost or slippage modeling
- Short backtest window — a longer history across multiple vol regimes would improve statistical validity
- Strike selection could be improved by adding a minimum spread width filter to avoid legs converging